# **4. 객체탐지의 원리** — IoU · NMS · mAP 직접 구현


---





- 목표
    - ① 경계 상자(bbox) 표현 변환
    - ② **IoU / GIoU / DIoU / CIoU** 직접 구현
    - ③ **NMS** 직접 구현
    - ④ **Precision–Recall–AP–mAP** 계산
    - ⑤ 실제 탐지기(YOLO11 · YOLO26 · RT-DETR) 비교
- 검증 방법
    - 직접 만든 함수 결과를 `torchvision.ops` 공식 구현과 비교 |
- 실행 환경
    - Colab 또는 PC, CPU/GPU
    



---


**(코랩에서)한글 폰트 지정하는 방법**

In [ ]:
# 코랩에서 실행
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import shutil
import warnings
import logging

# ── 1. 나눔 폰트 설치 ──────────────────────────────────────────
!apt-get install -y fonts-nanum > /dev/null 2>&1

# ── 2. matplotlib 캐시 디렉터리 삭제 ──────────────────────────
shutil.rmtree(mpl.get_cachedir(), ignore_errors=True)

# ── 3. 폰트 등록 및 유니코드 마이너스 경고 방지 ──────────────────
font_path = '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf'
fm.fontManager.addfont(font_path)
font_name = fm.FontProperties(fname=font_path).get_name()
plt.rcParams['font.family'] = font_name
plt.rcParams['font.size'] = 12
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (6, 4)

# 수학 기호(마이너스 등)를 일반 텍스트로 처리하도록 설정하여 폰트 에러 차단
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.default'] = 'regular'

# mathtext 및 font_manager 관련 폰트 결함 경고 완전히 무시 설정
logging.getLogger('matplotlib.mathtext').setLevel(logging.ERROR)
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib.font_manager')



---


## 0. 환경 설정

In [ ]:
# [Colab 명령] 한글 폰트: !apt-get -qq install -y fonts-nanum > /dev/null ; 이후 세션 다시 시작
# [Colab 명령] 6절용: !pip -q install ultralytics
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.font_manager as fm
import torch
from torchvision import ops

def set_korean_font():
    installed = {f.name for f in fm.fontManager.ttflist}
    for name in ["Malgun Gothic", "NanumGothic", "AppleGothic", "Noto Sans CJK KR"]:
        if name in installed:
            plt.rcParams["font.family"] = name; break
    plt.rcParams["axes.unicode_minus"] = False
set_korean_font()
print("torch", torch.__version__)

## **4-1. 경계 상자(Bounding Box) 표현 4가지**

| 형식 | 값 | 사용처 |
|---|---|---|
| xyxy | (x1, y1, x2, y2) 좌상단·우하단 | Pascal VOC, torchvision |
| xywh | (x, y, w, h) 좌상단 + 크기 | COCO, AI Hub |
| cxcywh (정규화) | (cx, cy, w, h) ÷ 이미지 크기 → 0~1 | **YOLO 라벨 txt** |
| 꼭짓점 8개 | (x1,y1,…,x4,y4) | 회전 상자(OBB) |

In [ ]:
def xyxy_to_xywh(b):
    x1, y1, x2, y2 = b
    return (x1, y1, x2 - x1, y2 - y1)

def xywh_to_xyxy(b):
    x, y, w, h = b
    return (x, y, x + w, y + h)

def xyxy_to_yolo(b, W, H):
    """xyxy(픽셀) → YOLO 정규화 (cx, cy, w, h)"""
    x1, y1, x2, y2 = b
    return ((x1 + x2) / 2 / W, (y1 + y2) / 2 / H, (x2 - x1) / W, (y2 - y1) / H)

def yolo_to_xyxy(b, W, H):
    cx, cy, w, h = b
    return ((cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H)


box = (50, 50, 150, 150)
W, H = 640, 480
y = xyxy_to_yolo(box, W, H)
print("xyxy  :", box)
print("xywh  :", xyxy_to_xywh(box))
print("YOLO  :", tuple(round(v, 4) for v in y), " ← 라벨 txt 한 줄: '클래스번호 cx cy w h'")
print("복원  :", tuple(round(v, 1) for v in yolo_to_xyxy(y, W, H)))
print("torchvision box_convert 검증:", ops.box_convert(torch.tensor([box], dtype=torch.float), "xyxy", "cxcywh").tolist())

## **4-2. IoU** (Intersection over Union)

$$\text{IoU}(A,B)=\frac{|A\cap B|}{|A\cup B|}=\frac{\text{겹친 넓이}}{A\text{넓이}+B\text{넓이}-\text{겹친 넓이}}$$

In [ ]:
def iou(a, b):
    """두 상자(xyxy)의 IoU"""
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)           # 겹치지 않으면 0
    area = lambda r: (r[2] - r[0]) * (r[3] - r[1])
    return inter / (area(a) + area(b) - inter)

A = (50, 50, 150, 150)       # 정답 상자 (100×100)
B = (100, 100, 200, 200)     # 예측 상자 (100×100)
print(f"겹친 넓이 = 50×50 = 2,500 | 합집합 = 10,000 + 10,000 − 2,500 = 17,500")
print(f"IoU(직접) = {iou(A, B):.4f} | IoU(torchvision) = {ops.box_iou(torch.tensor([A], dtype=torch.float), torch.tensor([B], dtype=torch.float)).item():.4f}")

fig, ax = plt.subplots(figsize=(4, 4))
for b, c, t in [(A, "green", "정답"), (B, "red", "예측")]:
    ax.add_patch(patches.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, ec=c, lw=2, label=t))
ax.add_patch(patches.Rectangle((100, 100), 50, 50, color="orange", alpha=.5, label="교집합"))
ax.set_xlim(0, 250)
ax.set_ylim(250, 0)
ax.legend()
ax.set_title(f"IoU = {iou(A, B):.3f}")
plt.show()

## **4-3. IoU의 한계와 개선** — GIoU(2019) · DIoU · CIoU(2020)

| 지표 | 추가 항 | 해결하는 문제 |
|---|---|---|
| IoU | – | 겹치지 않으면 항상 0 → 손실의 기울기 0 |
| GIoU | $-\frac{|C\setminus(A\cup B)|}{|C|}$ (C: 둘을 감싸는 최소 상자) | 떨어진 정도를 반영 (−1 ~ 1) |
| DIoU | $-\frac{\rho^2(\text{중심})}{c^2}$ (c: C의 대각선) | 중심 거리를 직접 줄임 → 빠른 수렴 |
| CIoU | DIoU $-\alpha v$ (v: 가로세로비 차이) | 모양(비율)까지 맞춤 → YOLO 계열 상자 손실에 사용 |

In [ ]:
def iou_family(a, b):
    a, b = np.array(a, float), np.array(b, float)
    inter = max(0, min(a[2], b[2]) - max(a[0], b[0])) * max(0, min(a[3], b[3]) - max(a[1], b[1]))
    wa, ha, wb, hb = a[2] - a[0], a[3] - a[1], b[2] - b[0], b[3] - b[1]
    union = wa * ha + wb * hb - inter
    i = inter / union
    cx1, cy1 = min(a[0], b[0]), min(a[1], b[1])
    cx2, cy2 = max(a[2], b[2]), max(a[3], b[3])
    c_area = (cx2 - cx1) * (cy2 - cy1)
    giou = i - (c_area - union) / c_area
    rho2 = ((a[0] + a[2]) / 2 - (b[0] + b[2]) / 2) ** 2 + ((a[1] + a[3]) / 2 - (b[1] + b[3]) / 2) ** 2
    c2 = (cx2 - cx1) ** 2 + (cy2 - cy1) ** 2
    diou = i - rho2 / c2
    v = 4 / np.pi ** 2 * (np.arctan(wa / ha) - np.arctan(wb / hb)) ** 2
    alpha = v / (1 - i + v + 1e-9)
    ciou = diou - alpha * v
    return i, giou, diou, ciou

gt = (100, 100, 200, 200)
cases = {"많이 겹침": (110, 110, 210, 210),
         "살짝 겹침": (170, 170, 270, 270),
         "가까이 떨어짐": (210, 100, 310, 200),
         "멀리 떨어짐": (400, 100, 500, 200),
         "모양 다름": (100, 130, 200, 170)}
print(f"{'상황':<10}{'IoU':>8}{'GIoU':>8}{'DIoU':>8}{'CIoU':>8}   | torchvision GIoU/DIoU/CIoU")
for name, pb in cases.items():
    i, g, d, c = iou_family(gt, pb)
    t1, t2 = torch.tensor([gt], dtype=torch.float), torch.tensor([pb], dtype=torch.float)
    tg, td, tc = ops.generalized_box_iou(t1, t2).item(), ops.distance_box_iou(t1, t2).item(), ops.complete_box_iou(t1, t2).item()
    print(f"{name:<10}{i:>8.3f}{g:>8.3f}{d:>8.3f}{c:>8.3f}   | {tg:.3f} / {td:.3f} / {tc:.3f}")

**관찰** : '가까이'와 '멀리'는 IoU가 모두 0이지만, GIoU·DIoU는 멀수록 더 작아집니다 → **학습 신호가 생깁니다.**

## **4-4. NMS (Non-Maximum Suppression)** — 중복 상자 지우기

| 단계 | 동작 |
|---|---|
| ① | 신뢰도(score) 순으로 정렬 |
| ② | 가장 높은 상자를 '채택' |
| ③ | 채택한 상자와 IoU가 임계값(예: 0.5) 이상인 상자는 제거 |
| ④ | 남은 상자로 ②~③ 반복 |

In [ ]:
def nms(boxes, scores, iou_thr=0.5):
    order = np.argsort(scores)[::-1]
    keep = []
    while len(order) > 0:
        i = order[0]; keep.append(int(i))
        rest = [j for j in order[1:] if iou(boxes[i], boxes[j]) < iou_thr]
        order = np.array(rest, dtype=int)
    return keep

boxes = np.array([[100, 100, 210, 220], [110, 105, 220, 230], [95, 90, 200, 210],     # 물체 1 근처
                  [300, 120, 400, 260], [310, 130, 405, 255], [120, 115, 230, 240]], float)
scores = np.array([0.92, 0.85, 0.78, 0.88, 0.60, 0.40])
keep = nms(boxes, scores, 0.5)
keep_tv = ops.nms(torch.tensor(boxes), torch.tensor(scores), 0.5).tolist()
print("직접 구현 NMS 채택:", keep, "| torchvision.ops.nms:", keep_tv)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, ids, t in [(axes[0], range(len(boxes)), "NMS 전 (6개)"), (axes[1], keep, f"NMS 후 ({len(keep)}개)")]:
    for i in ids:
        b = boxes[i]
        ax.add_patch(patches.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, lw=2,
                                       ec=plt.cm.tab10(i)))
        ax.text(b[0], b[1] - 4, f"{scores[i]:.2f}", color=plt.cm.tab10(i), fontsize=9)
    ax.set_xlim(50, 450); ax.set_ylim(300, 50); ax.set_title(t)
plt.tight_layout(); plt.show()

## **4-5. 평가 지표 — Precision · Recall · AP · mAP**

| 용어 | 정의 (탐지) |
|---|---|
| TP | 정답과 IoU ≥ 임계값이고, 그 정답을 처음 맞힌 예측 |
| FP | 맞힌 정답이 없거나 이미 맞힌 정답과 중복된 예측 |
| Precision | TP / (TP + FP) — 찾은 것 중 맞은 비율 |
| Recall | TP / 전체 정답 수 — 정답 중 찾은 비율 |
| AP | Precision–Recall 곡선 아래 면적 (클래스 1개) |
| mAP50 | IoU 0.5 기준 AP의 클래스 평균 (Pascal VOC 방식) |
| mAP50-95 | IoU 0.50, 0.55, …, 0.95 (10개) 평균 (COCO 기본 지표) |

In [ ]:
# 이미지 3장, 정답 5개, 예측 8개 (한 클래스) — 손으로 따라갈 수 있는 크기
gts = {0: [(10, 10, 60, 60), (100, 100, 160, 170)],
       1: [(30, 40, 90, 120)],
       2: [(50, 50, 120, 110), (200, 30, 260, 90)]}
dets = [  # (이미지, score, 상자)
    (0, 0.95, (12, 12, 62, 58)), (2, 0.91, (52, 48, 118, 112)), (1, 0.88, (28, 45, 95, 118)),
    (0, 0.80, (105, 110, 150, 175)), (2, 0.70, (55, 52, 125, 108)), (0, 0.55, (200, 200, 240, 240)),
    (2, 0.45, (215, 40, 270, 100)), (1, 0.30, (60, 60, 120, 140))]

def evaluate_ap(gts, dets, iou_thr=0.5, verbose=False):
    n_gt = sum(len(v) for v in gts.values())
    used = {k: [False] * len(v) for k, v in gts.items()}
    tp = []
    for img, s, b in sorted(dets, key=lambda d: -d[1]):          # 신뢰도 높은 순
        ious = [iou(b, g) for g in gts[img]]
        j = int(np.argmax(ious)) if ious else -1
        hit = j >= 0 and ious[j] >= iou_thr and not used[img][j]
        if hit: used[img][j] = True
        tp.append(1 if hit else 0)
        if verbose:
            print(f"  img{img} score {s:.2f}  최대IoU {max(ious) if ious else 0:.2f} → {'TP' if hit else 'FP'}")
    tp = np.array(tp); fp = 1 - tp
    recall = np.cumsum(tp) / n_gt
    precision = np.cumsum(tp) / (np.cumsum(tp) + np.cumsum(fp))
    # COCO 방식: 정밀도를 오른쪽에서부터 단조 감소로 만든 뒤 recall 101개 지점에서 평균
    mprec = np.maximum.accumulate(precision[::-1])[::-1]
    rs = np.linspace(0, 1, 101)
    ap = np.mean([mprec[recall >= r].max() if (recall >= r).any() else 0 for r in rs])
    return ap, precision, recall

ap50, prec, rec = evaluate_ap(gts, dets, 0.5, verbose=True)
print(f"\nAP@0.5 = {ap50:.3f}")
aps = [evaluate_ap(gts, dets, t)[0] for t in np.arange(0.5, 0.96, 0.05)]
print("IoU 임계값별 AP:", [round(float(a), 2) for a in aps])
print(f"AP@[.5:.95] = {np.mean(aps):.3f}   ← 임계값이 엄격할수록 AP가 떨어짐")

plt.figure(figsize=(4.5, 3.6))
plt.step(rec, prec, where="post", marker="o")
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.ylim(0, 1.05); plt.xlim(0, 1.0)
plt.title(f"PR 곡선 (AP@0.5 = {ap50:.2f})"); plt.grid(alpha=.3); plt.show()

## **4-6. 실제 탐지기 비교** — YOLO11n(NMS 사용) vs YOLO26n(NMS-free) vs RT-DETR-l(트랜스포머)

같은 사진(ultralytics 패키지에 들어 있는 bus.jpg)으로 CPU 추론 시간과 결과를 비교합니다.

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO, RTDETR
from ultralytics.utils import ASSETS          # 패키지 내장 예제 이미지 폴더
import time
import matplotlib.pyplot as plt
import torch
from torchvision import ops

img_path = str(ASSETS / "bus.jpg")

detectors = {"YOLO11n": YOLO("yolo11n.pt"), "YOLO26n": YOLO("yolo26n.pt"), "RT-DETR-l": RTDETR("rtdetr-l.pt")}
rows, plots = [], []
for name, m in detectors.items():
    m.predict(img_path, verbose=False)                          # 첫 실행(워밍업) 제외
    t0 = time.time()
    for _ in range(3):
        r = m.predict(img_path, conf=0.25, verbose=False)[0]
    ms = (time.time() - t0) / 3 * 1000
    n_param = sum(p.numel() for p in m.model.parameters()) / 1e6
    names = [r.names[int(c)] for c in r.boxes.cls]
    rows.append((name, n_param, ms, len(r.boxes), ", ".join(sorted(set(names)))))
    plots.append((name, r.plot()[:, :, ::-1]))                  # BGR → RGB

print(f"{'모델':<10}{'파라미터(M)':>12}{'CPU 추론(ms)':>14}{'상자 수':>8}  클래스")
for n, p, ms, k, cls in rows:
    print(f"{n:<10}{p:>12.1f}{ms:>14.0f}{k:>8}  {cls}")

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
for ax, (n, im) in zip(axes, plots):
    ax.imshow(im); ax.set_title(n); ax.axis("off")
plt.tight_layout(); plt.show()

### 임계값(conf · iou)이 결과를 바꾼다 — YOLO11n

In [ ]:
m = detectors["YOLO11n"]
print(f"{'conf':>6}{'iou(NMS)':>10}{'상자 수':>8}")
for conf, iou_thr in [(0.5, 0.7), (0.25, 0.7), (0.05, 0.7), (0.05, 0.95)]:
    r = m.predict(img_path, conf=conf, iou=iou_thr, verbose=False)[0]
    print(f"{conf:>6}{iou_thr:>10}{len(r.boxes):>8}")

**관찰** : conf를 낮추면 상자가 늘고(Recall↑, Precision↓), NMS의 iou를 0.95로 올리면 겹친 중복 상자가 거의 지워지지 않습니다.

## **4-7. [선택] 2-stage 탐지기** — torchvision Faster R-CNN

`RUN_FRCNN = True` 로 바꾸면 Faster R-CNN(Ren et al., 2015)으로 같은 사진을 탐지합니다. 가중치 약 160MB, **GPU 권장**.
- [주의!] GPU에서 실행하기

In [ ]:
RUN_FRCNN = False
if RUN_FRCNN:
    from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
    from torchvision.io import read_image
    w = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
    frcnn = fasterrcnn_resnet50_fpn_v2(weights=w).eval()
    x = w.transforms()(read_image(img_path))
    t0 = time.time()
    with torch.no_grad():
        out = frcnn([x])[0]
    keep = out["scores"] > 0.5
    print(f"Faster R-CNN: {keep.sum().item()}개, {1000 * (time.time() - t0):.0f} ms",
          [w.meta["categories"][i] for i in out["labels"][keep]])